### Piping a Prompt, Model, and an Output Parser

In [1]:
from dotenv import load_dotenv
import os

# Don't need to specify full path if notbook file and .env file are on the same leve in the project structure
load_dotenv("../.env")

True

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

In [3]:
list_instructions = CommaSeparatedListOutputParser().get_format_instructions()

In [4]:
list_instructions

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [15]:
chat_template = ChatPromptTemplate.from_messages([
    ('human', "I have adopted a {pet}. Could you suggest three {pet} names? \n" + list_instructions)
])

In [16]:
"""
Notice that we didn't need to import and create an instance of this class (HumanMessagePromptTemplate). Conveniently,
it was done for us.
"""
chat_template

ChatPromptTemplate(input_variables=['pet'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['pet'], input_types={}, partial_variables={}, template='I have adopted a {pet}. Could you suggest three {pet} names? \nYour response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'), additional_kwargs={})])

In [17]:
print(chat_template.messages[0].prompt.template)

I have adopted a {pet}. Could you suggest three {pet} names? 
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [18]:
chat = ChatOpenAI(
    model="gpt-5",
    seed=365, 
    temperature=0
)

In [19]:
list_output_parser = CommaSeparatedListOutputParser()

In [21]:
chain = chat_template | chat | list_output_parser

In [ ]:
chain.invoke({
    'pet': 'dog'
})

### Batching

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [23]:
chat_template = ChatPromptTemplate.from_messages([
    ('human', "I've recently adopted a {pet} which is a {breed}. Could you suggest several training tips?")
])

In [24]:
chat = ChatOpenAI(model_name = 'gpt-4',
                  seed=365,
                  temperature = 0,
                  max_tokens = 100)

In [25]:
chain = chat_template | chat

In [ ]:
chain.invoke({
    "pet": "dog",
    "breed": "shepherd"
})

![invoke](../images/image-5.png)

In [ ]:
chain.batch([
    {"pet": "dog", "breed": "shepherd"},
    {"pet": "dragon", "breed": "night fury"}
])

### Stream

In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [28]:
chat_template = ChatPromptTemplate.from_messages([
    ('human', "I've recently adopted a {pet} which is a {breed}. Could you suggest several training tips?")
])

In [33]:
chat = ChatOpenAI(model_name = 'gpt-5',
                  seed=365,
                  temperature = 0,
                  max_tokens = 100)

In [30]:
chain = chat_template | chat

In [ ]:
response = chain.stream({
    "pet": "dog",
    "breed": "shepherd"
})

In [ ]:
for i in response:
    print(i.content)

### The Runnable and RunnableSequence Classes

In [31]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [32]:
chat_template = ChatPromptTemplate.from_messages([
    ('human', "I've recently adopted a {pet} which is a {breed}. Could you suggest several training tips?")
])

In [ ]:
chat = ChatOpenAI(model_name = 'gpt-5',
                  seed=365,
                  temperature = 0,
                  max_tokens = 100)

In [35]:
chain = chat_template | chat

In [36]:
type(chain)

langchain_core.runnables.base.RunnableSequence

### Piping Chains and the RunnablePassthrough Class

In [39]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [40]:
RunnablePassthrough().invoke('hi')

'hi'

In [ ]:
# The output is the same list with no modifications. 
RunnablePassthrough().invoke([1, 3, 4])

[1, 3, 4]

![Roadmap](../images/image-6.png)

In [42]:
chat_template_tools = ChatPromptTemplate.from_template("""
What are the five most important tools a {job tile} needs?
Answer only by listing the tools. 
""")

![placeholder](../images/image-7.png)

In [ ]:
"""
Notice that by default, this new method (ChatPromptTemplate.from_template()) assumes that the template we passed
as an argument is a human message prompt template.
"""
chat_template_tools

ChatPromptTemplate(input_variables=['job tile'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['job tile'], input_types={}, partial_variables={}, template='\nWhat are the five most important tools a {job tile} needs?\nAnswer only by listing the tools. \n'), additional_kwargs={})])

In [44]:
chat_template_strategy = ChatPromptTemplate.from_template('''
Considering the tools provided, develop a strategy for effectively learning and mastering them:
{tools}
''')

In [ ]:
chat = ChatOpenAI(model_name = 'gpt-5',
                  seed=365,
                  temperature = 0)

In [46]:
str_output_parser = StrOutputParser()

In [49]:
chain_tools = chat_template_tools | chat | str_output_parser | { "tools": RunnablePassthrough()}

In [48]:
chain_strategy = chat_template_strategy | chat | str_output_parser

In [50]:
chain_combined = chain_tools | chain_strategy

In [ ]:
chain_combined.invoke({
    'job title': 'data scientist'
})

![Visualize](../images/image-8.png)

In [52]:
# uv add grandalf
chain_combined.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
     +-------------+       
     | Passthrough |       
     +-------------+       
            *              
            *              
            *       

### RunnableParallel

In [54]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [55]:
chat_template_books = ChatPromptTemplate.from_template(
    """
Suggest three of the best intermediate-level {programming language} books.
Answer only by listing the books.
"""
)

chat_template_projects = ChatPromptTemplate.from_template(
    """
Suggest three interesting {programming language} projects suitable for intermediate-level programmers.
Answer only by listing the projects.
"""
)

In [56]:
chat = ChatOpenAI(model_name = 'gpt-5',
                  seed=365,
                  temperature = 0)

In [57]:
str_output_parser = StrOutputParser()

In [58]:
chain_books = chat_template_books | chat | str_output_parser
chain_projects = chat_template_projects | chat | str_output_parser

In [61]:
chain_parallel = RunnableParallel({
    # Key names can be anything
    'books': chain_books,
    'projects': chain_projects
})

In [ ]:
chain_parallel.invoke({
    "programming language": "python"
})

In [62]:
chain_parallel.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

### Piping a RunnableParallel with Other Runnables

![runnable](../images/image-9.png)
![runnable](../images/image-10.png)

In [63]:
chat_template_time = ChatPromptTemplate.from_template(
    """
I'm an intermediate level programmer.

Consider the following literature:
{books}

Also, consider the following projects:
{projects}

Roughly how much time would it take me to complete the literature and the projects?
"""
)

In [64]:
chain_time = (
    RunnableParallel({
    # Key names can be anything
    'books': chain_books,
    'projects': chain_projects
    }) 
    | chat_template_time 
    | chat 
    | str_output_parser
)

In [ ]:
chain_time.invoke({
    "programming language": "python"
})

In [65]:
chain_time.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

Another way to construct parallel runnable without using RunnableParallel Class
![paralle](../images/image-11.png)

### Runnable Lambda

![lambda](../images/image-12.png)

In [1]:
find_sum = lambda x: sum(x)

In [3]:
find_sum([1, 2, 5])

8

In [4]:
find_square = lambda x: x**2

In [5]:
find_square(8)

64

In [6]:
from langchain_core.runnables import RunnableLambda

In [8]:
"""
Of course, we can pass the find_sum variable itself. But this will defy the purpose of lambda function whose primary
benefit is that it can be defined without a name. Lambda functions are anonymous functions.
"""
runnable_sum = RunnableLambda(lambda x: sum(x))

In [9]:
runnable_sum.invoke([1, 2, 5])

8

In [10]:
runnable_square = RunnableLambda(lambda x: x**2)

In [11]:
runnable_square.invoke(8)

64

In [12]:
chain = runnable_sum | runnable_square

In [13]:
chain.invoke([1, 2, 5])

64

In [14]:
chain.get_graph().print_ascii()

+-------------+  
| LambdaInput |  
+-------------+  
        *        
        *        
        *        
   +--------+    
   | Lambda |    
   +--------+    
        *        
        *        
        *        
   +--------+    
   | Lambda |    
   +--------+    
        *        
        *        
        *        
+--------------+ 
| LambdaOutput | 
+--------------+ 


### The @chain Decorator

In [20]:
from langchain_core.runnables import RunnableLambda, chain

In [16]:
def find_sum(x):
    return sum(x)

def find_square(x):
    return x**2

In [ ]:
"""
We've created the lesson example without using lambda function.
"""
chain1 = RunnableLambda(find_sum) | RunnableLambda(find_square)

In [18]:
chain1.invoke([1, 2, 5])

64

In [22]:
@chain
def runnable_sum(x):
    return sum(x)

@chain
def runnable_square(x):
    return x**2

In [24]:
type(runnable_sum)

langchain_core.runnables.base.RunnableLambda

In [25]:
chain2 = runnable_sum | runnable_square

In [26]:
chain2.invoke([1, 2, 5])

64